In [ ]:
from pathlib import Path
import sys

import polars as pl

# Resolve the repository root so the notebook works both when launched from the
# repository root and when launched from the notebooks/ directory.
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

# Use the BioTrove processing package that lives in this repository under
# src/smartrodent/biotrove_process instead of importing an external checkout.
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from smartrodent.biotrove_process import (
    GenImgTxtPair,
    GenShuffledChunks,
    GetImages,
    MetadataProcessor,
    load_config,
)
from smartrodent.dataprocessing import ImageFilter

In [ ]:
basepath = repo_root
basepath

In [ ]:
config_path = basepath / "configs" / "config_central_europe.json"

In [ ]:
datapath = basepath / "datasets" / "BioTrove" / "BioTrove"

# build the image dataset now

In [ ]:
%load_ext autoreload
%autoreload 2

config = load_config(config_path)

In [ ]:
%load_ext autoreload
%autoreload 2

params = config.get('metadata_processor_info', {})
mp = MetadataProcessor(**params)
mp.process_all_files()

# Step 2: generate shuffled chunks of metadata

filter parquet files first.
TODO: put this into the processing package, after making sure it's needed. 

In [ ]:
from pathlib import Path
import polars as pl

src = Path(config["metadata_processor_info"]["source_folder"])
filtered_dir = Path(config["filter_dir"])
categories = config["metadata_processor_info"]["categories"]
category_type = config["metadata_processor_info"]["category_type"]
filtered_dir.mkdir(parents=True, exist_ok=True)

for path in sorted(src.glob("*.parquet")):
    df = (
        pl.scan_parquet(path)
        .filter(pl.col(category_type).is_in(categories))
        .collect()
    )

    if df.height:
        df.write_parquet(filtered_dir / path.name)

then build chunks

In [ ]:
%load_ext autoreload
%autoreload 2

config = load_config(config_path)
params = config.get('metadata_filter_and_shuffle_info', {})
params["directory"] = str(filtered_dir)
params

In [ ]:
%load_ext autoreload
%autoreload 2
gen_shuffled_chunks = GenShuffledChunks(**params)
gen_shuffled_chunks.process_files()

# Step 3: Download images


In [ ]:
%load_ext autoreload
%autoreload 2

config = load_config(config_path)

In [ ]:
params = config.get('image_download_info', {})
params

In [ ]:
%load_ext autoreload
%autoreload 2

gi = GetImages(**params)
await gi.download_images()

# Step 4: Generate text pairs and create tar files (optional)

In [ ]:
%load_ext autoreload
%autoreload 2
config = load_config(config_path)
params = config.get('img_text_gen_info', {})
params

In [ ]:
%load_ext autoreload
%autoreload 2
textgen = GenImgTxtPair(**params)
textgen.create_image_text_pairs()

## Step 5: Generate classifier dataset

In [ ]:
import shutil

In [ ]:
img_dir = config["img_dir"]
Path(img_dir).mkdir(exist_ok=True, parents = True)

In [ ]:
params = config.get('img_text_gen_info', {})

for chunk in Path(params["img_folder"]).iterdir():
    if chunk.is_dir():
        jpgs = [c for c in Path(chunk).iterdir() if c.suffix == ".jpg"]
        scnames = {c.stem.split(".")[0]: c for c in Path(chunk).iterdir() if 'scientific_name' in str(c)}

        for jpg in jpgs:
            name = jpg.stem

            # find scientific name file
            scientific_name_file = scnames[name]

            with open(scientific_name_file, mode = "r") as scf:
                scientific_name = scf.read()[0:-1]

            species_img_path = (Path(img_dir) / scientific_name)
            species_img_path.mkdir(parents=True, exist_ok=True)

            print(scientific_name, ", ", jpg.name )
            # copy the jpg to the species_img_path
            shutil.copy2(
                jpg,
                str(species_img_path / Path(jpg).name)
            )



# Filtering using CLIP by OpenAI 

This is mostly based on https://colab.research.google.com/github/openai/clip/blob/master/notebooks/Interacting_with_CLIP.ipynb#scrollTo=0BpdJkdBssk9

In [ ]:
import clip
import numpy as np
import torch


check that you have cuda and stuff

In [ ]:
clip.available_models()

In [ ]:
healthy_prompt = "a live healthy looking wild animal"
dead_prompt = "a dead animal, roadkill, a skull or bones"
not_prompt = "not an animal at all"
rodent_prompt = " a mouse, rat or other rodent"

primary_prompts = [ not_prompt, rodent_prompt]
healthy_or_dead_prompt = [healthy_prompt, dead_prompt]

## Feed in example images

In [ ]:

from pathlib import Path
import random


In [ ]:
"""Deprecated local ImageFilter copy; use smartrodent.dataprocessing.ImageFilter.
    def __init__(
        self,
        model: str,
        prompts: list[str],
        id_tol: float = 0.02,
    ):

        self.model, self.preprocess = clip.load(model)
        self.id_tol = id_tol
        self.prompts = prompts

    def print(self):
        input_resolution = self.model.visual.input_resolution
        context_length = self.model.context_length
        vocab_size = self.model.vocab_size

        print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in self.model.parameters()]):,}")
        print("Input resolution:", input_resolution)
        print("Context length:", context_length)
        print("Vocab size:", vocab_size)



    def preprocess_images(self, impage_paths):
        original_images = []
        images = []
        for filename in impage_paths:
            image = Image.open(filename).convert("RGB")

            original_images.append(image)
            images.append(self.preprocess(image))
        return original_images, images

    def plot_images(self, images: list, figsize=(24, 28)):
        def get_optimal_squaring(n: float):
            if n < 0:
                raise RuntimeError("n < 0 forbidden")
            nx = int(sqrt(n))
            ny = nx
            nx_candidates = [nx - 1, nx, nx + 1]
            ny_candidates = [ny - 1, ny, ny + 1]

            n = 2 * len(images)
            frames = None
            for nx, ny in product(nx_candidates, ny_candidates):
                if nx * ny < len(images):
                    continue

                if nx * ny < n:
                    frames = (nx, ny)
                    n = nx * ny

            return frames

        nxny = get_optimal_squaring(len(images))

        if nxny is None:
            raise RuntimeError(
                f"Could not find optimal squaring for num. images {len(images)}"
            )

        else:
            nx, ny = nxny
            fig, axs = plt.subplots(nx, ny, figsize=figsize)
            axs = axs.flatten()
            for i, image in enumerate(images):
                axs[i].imshow(image)

            fig.tight_layout()

            return fig, axs

    def compute_similarity(self, images):
        image_input = torch.tensor(np.stack(images)).cuda()
        text_tokens = clip.tokenize(["This is " + desc for desc in self.prompts]).cuda()
        with torch.no_grad():
            image_features = self.model.encode_image(image_input).float()
            text_features = self.model.encode_text(text_tokens).float()

        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)

        similarity = text_features.cpu().detach() @ image_features.cpu().detach().T
        return similarity

    def filter_similarities(
        self,
        similarity: torch.Tensor,
    ):
        # filter cols of similarity matrix to see if the confidence difference is
        # larger than a given tolerance. those in which it is are good to go, the
        # others are collected in an 'undecided' heap that we don't really understand
        undecided = []
        decided = []
        undecided_idx = []
        decided_idx = []
        for i in range(similarity.shape[1]):
            minimum_col = similarity[:, i].min()
            maximum_col = similarity[:, i].max()

            if (maximum_col - minimum_col) <= self.id_tol:
                undecided.append(similarity[:, i].unsqueeze(1))
                undecided_idx.append(i)
            else:

                decided.append(similarity[:, i].unsqueeze(1))
                decided_idx.append(i)

        decided = torch.cat(decided, dim=1) if len(decided) > 0 else torch.empty(0)
        undecided = (
            torch.cat(undecided, dim=1) if len(undecided) > 0 else torch.empty(0)
        )

        return decided_idx, undecided_idx, decided, undecided

    def decisions(self, imgs, decided):
        best = decided.argmax(dim=0).tolist()
        labeled_data = {}
        for p in self.prompts:
            labeled_data[p] = []

        for i, img in zip(best, imgs):
            print("i: ", i, ", prompt: ", self.prompts[i])
            labeled_data[self.prompts[i]].append(img)

        return labeled_data

    def plot_sim_score(
        self,
        similarity: torch.Tensor,
        original_images: list[np.ndarray],
        figsize=(20, 14),
        yfontsize=18,
    ):
        if similarity.shape == (0,) or len(original_images) == 0:
            print("nothing to plot")
            return

        count = similarity.shape[0]

        plt.figure(figsize=figsize)
        plt.imshow(similarity, vmin=0.1, vmax=0.3)

        plt.yticks(range(count), self.prompts, fontsize=yfontsize)
        plt.xticks([])

        for i, image in enumerate(original_images):
            plt.imshow(
                image,
                extent=(i - 0.5, i + 0.5, -1.7, -0.7),
                origin="lower",
                aspect="equal",
                zorder=3,
                clip_on=False,
            )

        for x in range(similarity.shape[1]):
            for y in range(similarity.shape[0]):
                plt.text(
                    x,
                    y,
                    f"{similarity[y, x]:.2f}",
                    ha="center",
                    va="center",
                    size=12,
                )

        fig = plt.gcf()
        ax = plt.gca()
        for side in ["left", "top", "right", "bottom"]:
            ax.spines[side].set_visible(False)

        ax.set_xlim(-0.5, similarity.shape[1] - 0.5)
        ax.set_ylim(count - 0.5, -1.8)

        plt.title("Cosine similarity between text and image features", size=20)

        return fig, ax
"""
ImageFilter

In [ ]:
imgfilter = ImageFilter(model = 'RN50x16', prompts=primary_prompts, id_tol = 0.02)

In [ ]:
path = basepath / "datasets" / "biotrove-central-europe" / "imgs" / "Rattus rattus"

In [ ]:
rng_seed = 32456
sample_size = 10
rng = random.Random(rng_seed)
image_paths = rng.sample(list(path.iterdir()), sample_size)

In [ ]:
original_images, images = imgfilter.preprocess_images(image_paths)

In [ ]:
imgfilter.plot_images(original_images)

In [ ]:
similarity = imgfilter.compute_similarity(images)

In [ ]:
decided_idx, undecided_idx, decided, undecided = imgfilter.filter_similarities(
    similarity
)

In [ ]:
decided

In [ ]:
decided.argmax(dim=0)

In [ ]:
decided_idx

In [ ]:
imgfilter.plot_sim_score(decided, [original_images[i] for i in decided_idx], )

In [ ]:
imgfilter.plot_sim_score(undecided, [original_images[i] for i in undecided_idx], )

In [ ]:
rodent_images = [images[i] for i in decided_idx]
rodent_original_images = [original_images[i] for i in decided_idx]

In [ ]:
rodent_filter = ImageFilter('RN50x16', prompts = healthy_or_dead_prompt, id_tol =0.02)

In [ ]:
# original_images, images = rodent_filter.preprocess_images(rodent_images)
rodent_similarity = rodent_filter.compute_similarity(rodent_images)


In [ ]:
rodent_decided_idx, rodent_undecided_idx, rodent_decided, rodent_undecided = rodent_filter.filter_similarities(
    rodent_similarity
)

In [ ]:
rodent_filter.plot_sim_score(rodent_decided, [original_images[i] for i in rodent_decided_idx], )

In [ ]:
rodent_decided.argmax(dim=0)

In [ ]:
rodent_filter.plot_sim_score(rodent_undecided, [original_images[i] for i in undecided_idx], )

In [ ]:
labeled_data = rodent_filter.decisions(rodent_original_images, rodent_decided)

In [ ]:
labeled_data

In [ ]:
imgfilter.plot_images(labeled_data['a dead animal, roadkill, a skull or bones'])

In [ ]:
imgfilter.plot_images(labeled_data['a live healthy looking wild animal'])